# YOLO11s-Seg Architectural Segmentation Training & Registry Pipeline
### Phase 2: Model Training, ONNX Benchmarks, and Real-World Evaluation

This notebook orchestrates the model training, model export, ONNX validation/benchmarking, and real-world evaluation stages. Checkpoints are automatically backed up and recovered from Google Drive. Final outputs are cataloged into a versioned model registry folder.

## 1. Environment Setup & GPU Check

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies including ONNX, ONNX Runtime, and Ultralytics
!pip install -q ultralytics onnx onnxslim onnxruntime pandas opencv-python pyyaml

import os
import sys
import torch
from pathlib import Path
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU runtime! Please switch to GPU for YOLO training.")

## 2. Unzip and Verify Dataset Manifest

In [ ]:
import shutil
import hashlib
import json

zip_source = "/content/drive/MyDrive/BOM_Project/dataset_export.zip"
manifest_source = "/content/drive/MyDrive/BOM_Project/dataset_manifest.json"
local_dataset_dir = "/content/yolo_dataset"

if os.path.exists(zip_source):
    # Validate SHA-256 fingerprint if manifest exists
    if os.path.exists(manifest_source):
        with open(manifest_source, "r") as f:
            manifest = json.load(f)
        expected_hash = manifest.get("dataset_hash", "")
        
        # Calculate local hash
        sha256 = hashlib.sha256()
        with open(zip_source, "rb") as f:
            while chunk := f.read(8192):
                sha256.update(chunk)
        actual_hash = sha256.hexdigest()
        
        print(f"Manifest Date: {manifest.get('date', 'N/A')} | Version: {manifest.get('dataset_version', 'N/A')}")
        if actual_hash == expected_hash:
            print("Dataset integrity check: PASSED (SHA-256 matches manifest).")
        else: 
            print(f"WARNING: Dataset SHA-256 mismatch!\n Expected: {expected_hash}\n Calculated: {actual_hash}")
            
    print("Unzipping dataset files...")
    if os.path.exists(local_dataset_dir):
        shutil.rmtree(local_dataset_dir)
    shutil.unpack_archive(zip_source, local_dataset_dir, "zip")
    print("Dataset ready locally.")
else:
    print(f"ERROR: dataset_export.zip not found at {zip_source}. Make sure to generate it in Phase 1 first.")

## 3. YOLO Training Execution (with Direct Drive Saves, Resume, and Fallback)

In [ ]:
from ultralytics import YOLO
import os
import yaml
from pathlib import Path

drive_project_dir = "/content/drive/MyDrive/BOM_Project/models"
os.makedirs(drive_project_dir, exist_ok=True)
resume_path = os.path.join(drive_project_dir, "yolo11s_seg", "weights", "last.pt")
local_yaml_path = os.path.join(local_dataset_dir, "dataset.yaml")

DEBUG = False
model_type = "yolo11n-seg.pt" if DEBUG else "yolo11s-seg.pt"
epochs = 10 if DEBUG else 100

# Check auto-resume
resume_mode = False
if Path(resume_path).exists():
    print(f"Checkpoint detected at {resume_path}. Resuming training...")
    model = YOLO(resume_path)
    resume_mode = resume_path
else:
    print(f"Starting fresh training with baseline {model_type}...")
    model = YOLO(model_type)

# Set import paths
sys.path.append("/content/data_annotation")

try:
    print("Starting YOLO training...")
    results = model.train(
        data=local_yaml_path,
        epochs=epochs,
        imgsz=1024,
        batch=16,
        device=0,
        cache=True,
        amp=True,
        workers=4,
        patience=20,
        project=drive_project_dir,
        name="yolo11s_seg",
        plots=True,
        save=True,
        save_period=5,     # Save checkpoints every 5 epochs
        optimizer="AdamW",
        cos_lr=True,
        resume=resume_mode
    )
except Exception as e:
    print(f"Training error: {e}. Retrying with batch=8 memory fallback...")
    if resume_mode:
        model = YOLO(resume_path)
    results = model.train(
        data=local_yaml_path,
        epochs=epochs,
        imgsz=1024,
        batch=8,
        device=0,
        cache=True,
        amp=True,
        workers=4,
        patience=20,
        project=drive_project_dir,
        name="yolo11s_seg",
        plots=True,
        save=True,
        save_period=5,
        optimizer="AdamW",
        cos_lr=True,
        resume=resume_mode
    )

# Save training directory dynamically
run_dir = str(results.save_dir)
print("Training directory returned:", run_dir)
with open("/content/run_dir.txt", "w") as f:
    f.write(run_dir)

## 4. Audit Check: Find results.csv and best.pt

In [ ]:
from pathlib import Path

print("Searching for results.csv...")
for p in Path("/content").rglob("results.csv"):
    print(p)
    
print("\nSearching for best.pt...")
for p in Path("/content").rglob("best.pt"):
    print(p)

## 5. Compile Model Registry, ONNX Validation & Latency Benchmarks

In [ ]:
import time
import numpy as np
import onnxruntime as ort
import cv2
import json
import pickle
import shutil

if os.path.exists("/content/run_dir.txt"):
    with open("/content/run_dir.txt", "r") as f:
        run_dir = f.read().strip()
else:
    run_dir = os.path.join(drive_project_dir, "yolo11s_seg")

local_registry_dir = "models/v1"
run_weights = os.path.join(run_dir, "weights", "best.pt")

# Self-contained export pipeline function
def export_pipeline(weights_path, export_dir, img_size):
    print(f"Starting export pipeline for: {weights_path}")
    if not os.path.exists(weights_path):
        raise FileNotFoundError(f"Source weights file not found: {weights_path}")
    os.makedirs(export_dir, exist_ok=True)
    
    print("Loading YOLO model weights...")
    model = YOLO(weights_path)
    weights_dir = os.path.dirname(os.path.abspath(weights_path))
    weights_basename = os.path.splitext(os.path.basename(weights_path))[0]
    
    print("Exporting model to ONNX format...")
    onnx_path_src = model.export(format="onnx", imgsz=img_size, opset=12)
    print(f"ONNX export completed: {onnx_path_src}")
    
    print("Exporting model to TorchScript format...")
    ts_path_src = model.export(format="torchscript", imgsz=img_size)
    print(f"TorchScript export completed: {ts_path_src}")
    
    best_pt_dst = os.path.join(export_dir, "best.pt")
    best_onnx_dst = os.path.join(export_dir, "best.onnx")
    best_ts_dst = os.path.join(export_dir, "best.torchscript")
    pkl_dst = os.path.join(export_dir, "bom_model.pkl")
    
    print(f"Copying files to export folder: {export_dir}")
    shutil.copy2(weights_path, best_pt_dst)
    
    if onnx_path_src and os.path.exists(onnx_path_src):
        shutil.copy2(onnx_path_src, best_onnx_dst)
    else:
        onnx_fallback = os.path.join(weights_dir, f"{weights_basename}.onnx")
        if os.path.exists(onnx_fallback):
            shutil.copy2(onnx_fallback, best_onnx_dst)
            
    if ts_path_src and os.path.exists(ts_path_src):
        shutil.copy2(ts_path_src, best_ts_dst)
    else:
        ts_fallback = os.path.join(weights_dir, f"{weights_basename}.torchscript")
        if os.path.exists(ts_fallback):
            shutil.copy2(ts_fallback, best_ts_dst)
            
    # Create and save Metadata PKL
    metadata = {
        "classes": {
            0: "door",
            1: "window"
        },
        "img_size": img_size,
        "model_type": "YOLO11s-Seg",
        "epochs": 100,
        "dataset": "SVG Synthetic Floorplans",
        "weights": "best.pt"
    }
    with open(pkl_dst, "wb") as f:
        pickle.dump(metadata, f)
    print(f"Metadata dictionary successfully saved to: {pkl_dst}")

# Run exporter
export_pipeline(run_weights, local_registry_dir, 1024)

best_onnx = os.path.join(local_registry_dir, "best.onnx")
best_pt = os.path.join(local_registry_dir, "best.pt")

# ONNX Runtime Validation Checks
print("Starting ONNX runtime execution validation...")
try:
    session = ort.InferenceSession(best_onnx)
    input_name = session.get_inputs()[0].name
    output_names = [o.name for o in session.get_outputs()]
    
    # Verify with random input
    dummy_input = np.random.randn(1, 3, 1024, 1024).astype(np.float32)
    outputs = session.run(output_names, {input_name: dummy_input})
    print(f"ONNX Runtime validation: SUCCESS. Output shapes: {[o.shape for o in outputs]}")
except Exception as e:
    print(f"ONNX Runtime validation FAILED: {e}")

# Benchmark PyTorch vs ONNX Latency & FPS
sample_image = np.random.randint(0, 255, (1024, 1024, 3), dtype=np.uint8)
pt_latencies = []
onnx_latencies = []

# PyTorch benchmark
try:
    pt_model = YOLO(best_pt)
    for _ in range(5):  # warmup
        _ = pt_model(sample_image, imgsz=1024, verbose=False)
    for _ in range(20):
        t0 = time.perf_counter()
        _ = pt_model(sample_image, imgsz=1024, verbose=False)
        pt_latencies.append(time.perf_counter() - t0)
    pt_lat_ms = np.mean(pt_latencies) * 1000
    pt_fps = 1.0 / np.mean(pt_latencies)
except Exception as e:
    pt_lat_ms, pt_fps = -1.0, -1.0

# ONNX benchmark
try:
    input_data = cv2.resize(sample_image, (1024, 1024)).transpose(2, 0, 1)
    input_data = np.expand_dims(input_data, axis=0).astype(np.float32) / 255.0
    for _ in range(5):  # warmup
        _ = session.run(output_names, {input_name: input_data})
    for _ in range(20):
        t0 = time.perf_counter()
        _ = session.run(output_names, {input_name: input_data})
        onnx_latencies.append(time.perf_counter() - t0)
    onnx_lat_ms = np.mean(onnx_latencies) * 1000
    onnx_fps = 1.0 / np.mean(onnx_latencies)
except Exception as e:
    onnx_lat_ms, onnx_fps = -1.0, -1.0

# Save report
benchmarks = {
    "pytorch": {"latency_ms": pt_lat_ms, "fps": pt_fps},
    "onnx": {"latency_ms": onnx_lat_ms, "fps": onnx_fps}
}
bench_path = os.path.join(local_registry_dir, "inference_benchmark.json")
with open(bench_path, "w") as f:
    json.dump(benchmarks, f, indent=2)
print(f"Benchmarks stored inside model registry: {bench_path}")

## 6. Real-World Evaluation, BOM MAE Counts, and Failure Taxonomy

In [ ]:
import glob
import cv2
import numpy as np
from ultralytics import YOLO

hard_examples_root = os.path.join(local_registry_dir, "hard_examples")
missed_doors_dir = os.path.join(hard_examples_root, "missed_doors")
false_doors_dir = os.path.join(hard_examples_root, "false_doors")
partial_masks_dir = os.path.join(hard_examples_root, "partial_masks")
window_confusions_dir = os.path.join(hard_examples_root, "window_confusions")

for path in [missed_doors_dir, false_doors_dir, partial_masks_dir, window_confusions_dir]:
    os.makedirs(path, exist_ok=True)

best_pt = os.path.join(local_registry_dir, "best.pt")
model = YOLO(best_pt)

# Search paths for validation data
val_img_dir = None
val_lbl_dir = None

candidates = [
    ("/content/real_validation/images", "/content/real_validation/labels"),
    ("/content/data_annotation/good_ones_dataset/images/val", "/content/data_annotation/good_ones_dataset/labels/val"),
    ("/content/yolo_dataset/images/val", "/content/yolo_dataset/labels/val")
]
for c_img, c_lbl in candidates:
    if os.path.exists(c_img) and len(os.listdir(c_img)) > 0:
        val_img_dir = c_img
        val_lbl_dir = c_lbl
        break

def iou_boxes(b1, b2):
    xi1 = max(b1[0], b2[0])
    yi1 = max(b1[1], b2[1])
    xi2 = min(b1[2], b2[2])
    yi2 = min(b1[3], b2[3])
    
    inter = max(0.0, xi2 - xi1) * max(0.0, yi2 - yi1)
    a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
    a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
    union = a1 + a2 - inter
    return inter / union if union > 1e-5 else 0.0

if val_img_dir:
    eval_images = glob.glob(os.path.join(val_img_dir, "*"))
    eval_images = [e for e in eval_images if e.lower().endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Starting evaluation on: {val_img_dir} ({len(eval_images)} images)")
    
    total_gt_doors = 0
    total_gt_windows = 0
    total_pred_doors = 0
    total_pred_windows = 0
    door_mae_sum = 0.0
    window_mae_sum = 0.0
    labeled_files_count = 0
    
    for path in eval_images:
        basename = os.path.basename(path)
        plan_name = os.path.splitext(basename)[0]
        img = cv2.imread(path)
        h_img, w_img = img.shape[:2]
        
        # Load Ground Truth
        gt_boxes = []
        gt_lbl_path = os.path.join(val_lbl_dir, f"{plan_name}.txt") if val_lbl_dir else ""
        
        gt_doors = 0
        gt_windows = 0
        
        if os.path.exists(gt_lbl_path) and os.path.getsize(gt_lbl_path) > 0:
            with open(gt_lbl_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
if not parts: continue
cls = int(parts[0])
coords = [float(x) for x in parts[1:]]
xs = coords[0::2]
ys = coords[1::2]
x1 = min(xs) * w_img
y1 = min(ys) * h_img
x2 = max(xs) * w_img
y2 = max(ys) * h_img
gt_boxes.append((cls, [x1, y1, x2, y2]))
if cls == 0:
                        gt_doors += 1
elif cls == 1:
                        gt_windows += 1
            total_gt_doors += gt_doors
            total_gt_windows += gt_windows
            labeled_files_count += 1
            
        # Run Inference
        res = model.predict(source=path, imgsz=1024, conf=0.25, verbose=False)[0]
        pred_boxes = []
        pred_doors = 0
        pred_windows = 0
        
        if res.boxes is not None:
            for box in res.boxes:
                cls = int(box.cls[0].item())
                conf = float(box.conf[0].item())
                xyxy = box.xyxy[0].tolist()
                pred_boxes.append((cls, xyxy, conf))
                if cls == 0:
                    pred_doors += 1
                elif cls == 1:
                    pred_windows += 1
                    
        total_pred_doors += pred_doors
        total_pred_windows += pred_windows
        
        if os.path.exists(gt_lbl_path):
            door_mae_sum += abs(gt_doors - pred_doors)
            window_mae_sum += abs(gt_windows - pred_windows)
            
        # Run taxonomy checks
        matched_gt = set()
        matched_pred = set()
        
        for p_idx, (p_cls, p_box, p_conf) in enumerate(pred_boxes):
            best_iou = 0.0
            best_gt_idx = -1
            for g_idx, (g_cls, g_box) in enumerate(gt_boxes):
                if g_idx in matched_gt: continue
                iou = iou_boxes(p_box, g_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = g_idx
                    
            if best_gt_idx != -1 and best_iou > 0.3:
                g_cls, g_box = gt_boxes[best_gt_idx]
                matched_gt.add(best_gt_idx)
                matched_pred.add(p_idx)
                
                if p_cls != g_cls:
                    # Confused window and door
                    cv2.imwrite(os.path.join(window_confusions_dir, f"conf_{basename}"), img)
                elif best_iou < 0.5:
                    # Partial mask overlaps
                    cv2.imwrite(os.path.join(partial_masks_dir, f"partial_{basename}"), img)
                    
        for g_idx, (g_cls, g_box) in enumerate(gt_boxes):
            if g_idx not in matched_gt:
                if g_cls == 0:
                    # Missed door
                    cv2.imwrite(os.path.join(missed_doors_dir, f"missed_{basename}"), img)
                    
        for p_idx, (p_cls, p_box, p_conf) in enumerate(pred_boxes):
            if p_idx not in matched_pred:
                if p_cls == 0:
                    # False alarm door
                    cv2.imwrite(os.path.join(false_doors_dir, f"false_{basename}"), img)
                    
    door_mae = door_mae_sum / max(1, labeled_files_count)
    window_mae = window_mae_sum / max(1, labeled_files_count)
    
    # Save metrics JSON
    bom_metrics = {
        "eval_dataset": os.path.basename(val_img_dir),
        "total_images": len(eval_images),
        "total_gt_doors": total_gt_doors,
        "total_pred_doors": total_pred_doors,
        "total_gt_windows": total_gt_windows,
        "total_pred_windows": total_pred_windows,
        "door_count_mae": door_mae,
        "window_count_mae": window_mae
    }
    bom_path = os.path.join(local_registry_dir, "bom_metrics.json")
    with open(bom_path, "w") as f:
        json.dump(bom_metrics, f, indent=2)
        
    # Save domain gap analyzer report
    domain_gap = {
        "synthetic_vs_real_gap": {
            "door_mae": door_mae,
            "window_mae": window_mae,
            "door_count_ratio": float(total_pred_doors) / max(1, total_gt_doors),
            "window_count_ratio": float(total_pred_windows) / max(1, total_gt_windows),
            "domain_shift_index": float(abs(total_gt_doors - total_pred_doors) + abs(total_gt_windows - total_pred_windows)) / max(1, total_gt_doors + total_gt_windows)
        }
    }
    gap_path = os.path.join(local_registry_dir, "domain_gap_report.json")
    with open(gap_path, "w") as f:
        json.dump(domain_gap, f, indent=2)
        
    print(f"MAE Report: Doors={door_mae:.4f} | Windows={window_mae:.4f}")
    print("Failure taxonomy examples cataloged inside model registry.")
else:
    print("No images found for validation. Skipping MAE and Failure taxonomy calculation.")

## 7. Bundle Model Registry and Backup to Drive

In [ ]:
import pandas as pd
import json
import shutil

drive_project_base = "/content/drive/MyDrive/BOM_Project"

# 1. Extract results.csv and save to registry training summary
csv_path = os.path.join(run_dir, "results.csv")
summary = {}
if os.path.exists(csv_path):
    try:
        df = pd.read_csv(csv_path)
        final_row = df.iloc[-1]
        summary = {
            "dataset": "SVG Synthetic Floorplans",
            "epochs": len(df),
            "precision": float(final_row.get('metrics/precision(B)', 0.0)),
            "recall": float(final_row.get('metrics/recall(B)', 0.0)),
            "map50_box": float(final_row.get('metrics/mAP50(B)', 0.0)),
            "map5095_box": float(final_row.get('metrics/mAP50-95(B)', 0.0)),
            "map50_mask": float(final_row.get('metrics/mAP50(M)', 0.0)),
            "map5095_mask": float(final_row.get('metrics/mAP50-95(M)', 0.0))
        }
    except Exception as e:
        print(f"Warning: Failed to parse results.csv: {e}")
        
with open(os.path.join(local_registry_dir, "training_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("Training summary saved.")

# Self-contained packager function
def package_artifacts(export_dir, training_run_dir, dataset_dir, zip_dest_dir, drive_backup_dir):
    print("="*60)
    print("Starting packaging stage...")
    os.makedirs(zip_dest_dir, exist_ok=True)
    
    # Copy metrics and plots from training run directory to exports
    print(f"Copying metrics and curves from: {training_run_dir} to: {export_dir}")
    metrics_files = [
        "results.csv",
        "PR_curve.png",
        "F1_curve.png",
        "confusion_matrix.png",
        "args.yaml"
    ]
    for f_name in metrics_files:
        src = os.path.join(training_run_dir, f_name)
        dst = os.path.join(export_dir, f_name)
        if os.path.exists(src):
            shutil.copy2(src, dst)
            print(f" - Copied {f_name}")
            
    # Package exports/ to bom_training_bundle.zip
    bundle_zip_path = os.path.join(zip_dest_dir, "bom_training_bundle.zip")
    print(f"Creating training bundle: {bundle_zip_path}")
    if os.path.exists(bundle_zip_path):
        os.remove(bundle_zip_path)
    shutil.make_archive(
        base_name=os.path.join(zip_dest_dir, "bom_training_bundle"),
        format="zip",
        root_dir=export_dir
    )
    print(f"Successfully generated {bundle_zip_path}")
    
    # Package dataset to dataset_export.zip
    dataset_zip_path = os.path.join(zip_dest_dir, "dataset_export.zip")
    print(f"Creating dataset backup: {dataset_zip_path}")
    if os.path.exists(dataset_zip_path):
        os.remove(dataset_zip_path)
    if os.path.exists(dataset_dir):
        shutil.make_archive(
            base_name=os.path.join(zip_dest_dir, "dataset_export"),
            format="zip",
            root_dir=dataset_dir
        )
        print(f"Successfully generated {dataset_zip_path}")
        
    # Copy to Google Drive if available
    if drive_backup_dir:
        print(f"Google Drive target backup directory: {drive_backup_dir}")
        try:
            # Recursively copy the entire export_dir (e.g. models/v1) to Drive
            version_dir = os.path.basename(os.path.normpath(export_dir))
            drive_version_dir = os.path.join(drive_backup_dir, "models", version_dir)
            print(f" - Copying model registry recursively to Drive: {drive_version_dir}")
            shutil.copytree(export_dir, drive_version_dir, dirs_exist_ok=True)
            
            # Also back up the zip files to the root of Drive backup directory
            for zip_file in [bundle_zip_path, dataset_zip_path]:
                if os.path.exists(zip_file):
                    shutil.copy2(zip_file, os.path.join(drive_backup_dir, os.path.basename(zip_file)))
                    print(f" - Backed up zip file: {os.path.basename(zip_file)}")
            print("Google Drive backup completed successfully!")
        except Exception as e:
            print(f"Warning: Failed to back up files to Google Drive: {e}")
            
    print("Packaging Stage Completed Successfully!")

# Run packager
package_artifacts(local_registry_dir, run_dir, local_dataset_dir, ".", drive_project_base)
print("Model registry packaging and Drive synchronization complete!")